In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ast

In [4]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

movies = movies.merge(credits, on='title')

print(f"Loaded {len(movies)} movies.")
print(movies.columns.tolist())  

Loaded 4809 movies.
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'movie_id', 'cast', 'crew']


In [5]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

def extract_names(text):
    """
    Converts a JSON string like '[{"name": "Action"}, {"name": "Drama"}]'
    into a list ['Action', 'Drama']
    """
    try:
        items = ast.literal_eval(text)  
        return [item['name'] for item in items]
    except:
        return []

def extract_top_cast(text, max_cast=3):
    """
    Same as above, but only keeps the first 3 cast members.
    More cast = more noise for our algorithm.
    """
    try:
        items = ast.literal_eval(text)
        return [item['name'] for item in items[:max_cast]]
    except:
        return []

def extract_director(text):
    """
    The 'crew' column has everyone: director, producer, etc.
    We specifically want the person whose job is 'Director'.
    """
    try:
        items = ast.literal_eval(text)
        for item in items:
            if item['job'] == 'Director':
                return [item['name']]
        return []
    except:
        return []


movies['genres']   = movies['genres'].apply(extract_names)
movies['keywords'] = movies['keywords'].apply(extract_names)
movies['cast']     = movies['cast'].apply(extract_top_cast)
movies['crew']     = movies['crew'].apply(extract_director)

movies.dropna(subset=['overview'], inplace=True)

In [6]:
def collapse_spaces(words):
    """Joins a list of words, removing internal spaces."""
    return [w.replace(" ", "") for w in words]

movies['tags'] = (
    movies['overview'].apply(lambda x: x.split())
    + movies['genres'].apply(collapse_spaces) * 3      # genres matter most
    + movies['keywords'].apply(collapse_spaces) * 2    # keywords matter a lot
    + movies['cast'].apply(collapse_spaces)            # cast matters least
    + movies['crew'].apply(collapse_spaces) * 2        # director matters
)

# Convert the list of words back into a single string
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x).lower())

# Keep only the columns we need going forward
final = movies[['movie_id', 'title', 'tags', 'genres']].reset_index(drop=True)

print("\nSample tag for 'Avatar':")
print(final[final['title'] == 'Avatar']['tags'].values[0][:300])


Sample tag for 'Avatar':
in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction action adventure fantasy sciencefiction action adventure fantasy sciencefiction cult


In [7]:
print(final[final['title'] == 'Avatar']['tags'].values[0][:])

in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction action adventure fantasy sciencefiction action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron jamescameron


In [8]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

tfidf_matrix = vectorizer.fit_transform(final['tags'])

print(f"\nTF-IDF matrix shape: {tfidf_matrix.shape}")


TF-IDF matrix shape: (4806, 5000)


In [9]:
similarity = cosine_similarity(tfidf_matrix)

print(f"\nSimilarity matrix shape: {similarity.shape}")


Similarity matrix shape: (4806, 4806)


In [10]:
# dont run!!! it's the first edition

def recommend(movie_title, num_recommendations=10):
    """
    Given a movie title, returns a list of similar movies.
    
    How it works:
    1. Find the row index of the given movie
    2. Look up that row in the similarity matrix
    3. Sort all movies by similarity score (highest first)
    4. Return the top N (skipping index 0, which is the movie itself)
    """
    
    # Find the movie in our dataframe (case-insensitive search)
    matches = final[final['title'].str.lower() == movie_title.lower()]
    
    if matches.empty:
        print(f"❌ Movie '{movie_title}' not found!")
        # Suggest close matches
        close = final[final['title'].str.lower().str.contains(movie_title.lower())]
        if not close.empty:
            print("Did you mean one of these?")
            for t in close['title'].head(5):
                print(f"  - {t}")
        return []
    
    # Get the integer position (iloc index) of this movie
    idx = matches.index[0]
    
    # Get this movie's row from the similarity matrix
    # Result: a list of (movie_index, similarity_score) pairs
    sim_scores = list(enumerate(similarity[idx]))
    
    # Sort by similarity score, highest first
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Skip the first result (that's the movie itself, similarity = 1.0)
    sim_scores = sim_scores[1 : num_recommendations + 1]
    
    # Get the actual movie titles
    movie_indices = [i[0] for i in sim_scores]
    recommendations = final['title'].iloc[movie_indices].tolist()
    
    return recommendations

In [ ]:
# DONT RUN!! SECOND EDITION

def genre_overlap_score(genres_a, genres_b):
    if not genres_a or not genres_b:
        return 0.7
    set_a = set(genres_a)
    set_b = set(genres_b)
    overlap = len(set_a & set_b)
    union   = len(set_a | set_b)
    jaccard = overlap / union
    return 0.3 + (0.7 * jaccard)

def recommend(movie_title, num_recommendations=10):
    matches = final[final['title'].str.lower() == movie_title.lower()]
    
    if matches.empty:
        print(f"❌ Movie '{movie_title}' not found!")
        return []

    idx = matches.index[0]
    query_genres = final.iloc[idx]['genres']  # e.g. ['Romance', 'Comedy']

    # Build adjusted scores using genre overlap
    sim_scores = []
    for i, raw_score in enumerate(similarity[idx]):
        candidate_genres = final.iloc[i]['genres']
        bonus = genre_overlap_score(query_genres, candidate_genres)
        adjusted_score = raw_score * bonus
        sim_scores.append((i, adjusted_score))

    # Sort by adjusted score, skip self (index 0 after sort is the movie itself)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1 : num_recommendations + 1]

    movie_indices = [i[0] for i in sim_scores]
    return final['title'].iloc[movie_indices].tolist()

In [10]:
# Third edition (run this one)

INCOMPATIBLE_GENRES = [
    {'Horror', 'Thriller'},
    {'Romance', 'Comedy', 'Family'},
    {'Animation', 'Family'},
]

def genres_are_compatible(genres_a, genres_b):
    set_a = set(genres_a)
    set_b = set(genres_b)
    for group in INCOMPATIBLE_GENRES:
        a_in_group = bool(set_a & group)
        b_in_group = bool(set_b & group)
        if a_in_group != b_in_group:
            return False
    return True


def genre_overlap_score(genres_a, genres_b):
    if not genres_a or not genres_b:
        return 0.7
    set_a = set(genres_a)
    set_b = set(genres_b)
    overlap = len(set_a & set_b)
    union   = len(set_a | set_b)
    jaccard = overlap / union
    return 0.3 + (0.7 * jaccard)


def recommend(movie_title, num_recommendations=10):
    matches = final[final['title'].str.lower() == movie_title.lower()]

    if matches.empty:
        print(f"❌ Movie '{movie_title}' not found!")
        return []

    idx = matches.index[0]
    query_genres = final.iloc[idx]['genres']

    # Step 1 — Fix 4: adjust scores by genre overlap
    sim_scores = []
    for i, raw_score in enumerate(similarity[idx]):
        candidate_genres = final.iloc[i]['genres']
        bonus = genre_overlap_score(query_genres, candidate_genres)
        adjusted_score = raw_score * bonus
        sim_scores.append((i, adjusted_score))

    # Sort by adjusted score, skip self
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:]

    # Step 2 — Fix 2: filter out incompatible genres
    results = []
    for movie_idx, score in sim_scores:
        candidate_genres = final.iloc[movie_idx]['genres']
        if genres_are_compatible(query_genres, candidate_genres):
            results.append(final['title'].iloc[movie_idx])
        if len(results) >= num_recommendations:
            break

    return results

In [11]:
test_movies = ['The Dark Knight', 'About Time', 'Inception', 'Toy Story']

for movie in test_movies:
    print(f"\n🎬 Movies similar to '{movie}':")
    results = recommend(movie)
    for i, rec in enumerate(results, 1):
        print(f"  {i}. {rec}")


🎬 Movies similar to 'The Dark Knight':
  1. The Dark Knight Rises
  2. Point Blank
  3. In Too Deep
  4. Contraband
  5. Harsh Times
  6. Dead Man Down
  7. The Equalizer
  8. Nighthawks
  9. Exiled
  10. The Yards

🎬 Movies similar to 'About Time':
  1. Safety Not Guaranteed
  2. The Helix... Loaded
  3. Hot Tub Time Machine
  4. Igby Goes Down
  5. The Cookout
  6. Outside Providence
  7. Dinner for Schmucks
  8. Men in Black 3
  9. Fetching Cody
  10. Somewhere in Time

🎬 Movies similar to 'Inception':
  1. Congo
  2. Paycheck
  3. Switchback
  4. Sky Captain and the World of Tomorrow
  5. Street Fighter: The Legend of Chun-Li
  6. Knowing
  7. Star Trek: Generations
  8. Pandorum
  9. Star Trek III: The Search for Spock
  10. Cargo

🎬 Movies similar to 'Toy Story':
  1. Toy Story 3
  2. Toy Story 2
  3. Meet the Deedles
  4. Monster House
  5. Monsters, Inc.
  6. Garfield
  7. Spirit: Stallion of the Cimarron
  8. Free Birds
  9. Stuart Little 2
  10. The Lego Movie


In [13]:
import pickle

with open('similarity.pkl', 'wb') as f:
    pickle.dump(similarity, f)

final.to_csv('movies_clean.csv', index=False)

print("Saved similarity.pkl and movies_clean.csv ✅")

Saved similarity.pkl and movies_clean.csv ✅


i understood that tf-idf is not what suitable for result i want


now i'll use sentence embedding models